In [61]:
import numpy as np
from scipy.linalg import solve

In [62]:
def vasicek_pp_paths(r0, a, b, sigma, T, n_time, n_sim, phi):
    dt = T / n_time
    r = np.zeros((n_sim, n_time + 1))
    r[:, 0] = r0
    mean_reversion = np.exp(-a * dt)
    variance = (sigma**2 / (2 * a)) * (1 - np.exp(-2 * a * dt))
    for t in range(1, n_time + 1):
        z = np.random.normal(size=n_sim)
        x_t = (
            r[:, t - 1] * mean_reversion
            + b * (1 - mean_reversion)
            + np.sqrt(variance) * z
        )
        r[:, t] = x_t + phi(t * dt)
    return r

def price_zcb_vasicek_pp(r0, a, b, sigma, T, n_time, n_sim, phi):
    dt = T / n_time
    # simulate short-rate paths
    r = vasicek_pp_paths(r0, a, b, sigma, T, n_time, n_sim, phi)
    # trapezoidal or left Riemann sum (left is standard MC practice)
    integral = np.sum(r[:, :-1] * dt, axis=1)
    # discount factors
    discounts = np.exp(-integral)
    # Monte Carlo estimator
    return np.mean(discounts)

In [63]:
def matrix_L_vasicek_pp(Nr, r_grid, a, b, sigma, phi_t):
    dr = r_grid[1] - r_grid[0]
    # -----------------------------
    # discounting term: -(r + phi(t)) f
    # -----------------------------
    def L_disc():
        return -np.diag(r_grid + phi_t * np.ones(Nr))
    # -----------------------------
    # drift term: a(b - r) f_r
    # -----------------------------
    def L_r():
        L = np.zeros((Nr, Nr))
        for i in range(1, Nr - 1):
            drift = a * (b - r_grid[i])
            L[i, i - 1] = -drift / (2 * dr)
            L[i, i + 1] =  drift / (2 * dr)
        return L
    # -----------------------------
    # diffusion term
    # -----------------------------
    def L_rr():
        L = np.zeros((Nr, Nr))
        coeff = 0.5 * sigma**2 / dr**2
        for i in range(1, Nr - 1):
            L[i, i - 1] = coeff
            L[i, i]     = -2 * coeff
            L[i, i + 1] = coeff
        return L
    return L_disc() + L_r() + L_rr()

def zcb_price_cn_vasicek_pp(
    r0, a, b, sigma, T,
    phi,
    Nt=150, Nr=150, N=1.0, rmax=None
):
    if rmax is None:
        rmax = b + 5 * sigma + 3 * r0
    r_grid = np.linspace(0, rmax, Nr)
    dt = T / Nt
    I = np.eye(Nr)
    V = np.full(Nr, N)
    for n in reversed(range(Nt)):
        tau = T - n * dt
        L = matrix_L_vasicek_pp(Nr, r_grid, a, b, sigma, phi(tau))
        M1 = I - 0.5 * dt * L
        M2 = I + 0.5 * dt * L
        V = solve(M1, M2 @ V)
        # boundary conditions
        V[0] = N * np.exp(-r_grid[0] * tau)
        V[-1] = 0
    return np.interp(r0, r_grid, V)

In [64]:
r0 = 0.05
a = 0.2
b = 0.04
sigma = 0.01

T = 1.0
n_time = 200
n_sim = 20000

# lambda shift
phi = lambda t: 0.002 * np.exp(-0.5 * t)

In [65]:
price_zcb_vasicek_pp(r0, a, b, sigma, T, n_time, n_sim, phi)

0.8126151415716739

In [66]:
zcb_price_cn_vasicek_pp(r0, a, b, sigma, T, phi)

0.9506395660861343